In [ ]:
# NOTE: This script is used for a standard CB analysis with a fixed future_id and different strategy_ids.

## Load packages
from costs_benefits_ssp.cb_calculate import CostBenefits
import numpy as np
import pandas as pd 
import sys
import os 

from costs_benefits_ssp.model.cb_data_model import TXTable,CostFactor,TransformationCost,StrategyInteraction

import polars as pl

In [ ]:
ssp_data = pd.read_csv('C:\\Users\\pkane\\sspla\\ssp_louisiana\\jobs_data_prep\\data\\simulations\\data_for_LSU\\louisiana_leap.csv')
att_primary = pd.read_csv('C:\\Users\\pkane\\sspla\\ssp_louisiana\\jobs_data_prep\\data\\simulations\\data_for_LSU\\ATTRIBUTE_PRIMARY.csv')
att_strategy = pd.read_csv('C:\\Users\\pkane\\sspla\\ssp_louisiana\\jobs_data_prep\\data\\simulations\\data_for_LSU\\ATTRIBUTE_STRATEGY.csv')

In [ ]:
## Define base strategy
strategy_code_base = "BASE"

## Instantiate an object of the CostBenefits class
cb = CostBenefits(ssp_data, att_primary, att_strategy, strategy_code_base)

# Once the excel file has been updated, we can reload it to update the cost factors database
cb.load_cb_parameters('C:\\Users\\pkane\\sspla\\ssp_louisiana\\1000_runs_ensamble_postprocessing\\cb\\cb_cost_factors\\cb_config_params.xlsx')

# Compute System Costs
results_system = cb.compute_system_cost_for_all_strategies(verbose=False)

# Compute Technical Costs
results_tx = cb.compute_technical_cost_for_all_strategies(verbose=False)

# Combine results
results_all = pd.concat([results_system, results_tx], ignore_index = True)

#-------------POST PROCESS SIMULATION RESULTS---------------
# Post process interactions among strategies that affect the same variables
results_all_pp = cb.cb_process_interactions(results_all)

# SHIFT any stray costs incurred from 2015 to 2025 to 2025 and 2035
results_all_pp_shifted = cb.cb_shift_costs(results_all_pp)

results_all_pp_shifted.to_csv('C:\\Users\\pkane\\sspla\\ssp_louisiana\\jobs_data_prep\\data\\simulations\\data_for_LSU\\cb_louisiana_leap.csv', index = False)

In [ ]:
in_file = 'C:\\Users\\pkane\\sspla\\ssp_louisiana\\jobs_data_prep\\data\\simulations\\data_for_LSU\\cb_louisiana_leap.csv'
out_file = 'C:\\Users\\pkane\\sspla\\ssp_louisiana\\jobs_data_prep\\data\\simulations\\data_for_LSU\\cb_louisiana_leap_wide.csv'


# --- load ---
cb_data = pd.read_csv(in_file)

# --- split the 'variable' column into parts ---
# R made 5 columns: name, sector, cb_type, item_1, item_2
# Use n=4 so we get at most 5 pieces even if extra ':' appear later.
parts = cb_data["variable"].astype(str).str.split(":", n=4, expand=True)
parts.columns = ["name", "sector", "cb_type", "item_1", "item_2"]

cb_data = pd.concat([cb_data, parts], axis=1)

# --- scaling and year ---
cb_data["value"] = cb_data["value"] / 1e9
cb_data["Year"]  = cb_data["time_period"] + 2015

# --- aggregate (sum, skipping NaNs as in na.rm=TRUE) ---
group_cols = ["cb_type", "strategy_code", "future_id", "Year"]
cb_agg = (
    cb_data
    .groupby(group_cols, dropna=False, as_index=False)["value"]
    .sum()
    .rename(columns={"value": "Cumulative"})
)

# --- wide format (dcast) ---
wide_cb = (
    cb_agg
    .pivot_table(
        index=[ "future_id", "strategy_code", "Year"],
        columns="cb_type",
        values="Cumulative",
        aggfunc="sum"        # safe even if duplicates appear
        # , fill_value=0     # uncomment if you prefer 0 instead of NaN
    )
    .reset_index()
)

# If you prefer flat columns after pivot (remove the name from columns):
wide_cb.columns.name = None
print("unique future_id values:", wide_cb["future_id"].nunique())

# --- save ---
wide_cb.to_csv(out_file, index=False)
